# Super TikTakToe: the line-layer design in self-play on Google Colab

This trains a **new network from scratch**: the size of large (96 channels, 8 blocks) with **line layers** added
(design D5 of `src/test_designs.py`), with exactly the settings large had. It is compared with large's own log
at the same iterations. Large scored against medium: **14% at iteration 20, 28% at 40, 35% at 60, 43% at 80, 48% at 100**.
The network is saved as `alphazero_large_lines.pt` in `Super-TikTakToe-models` in your Google Drive; nothing of large is touched.

1. `alphazero_medium.pt` must be in that Drive folder (it is, from the training of large).
2. Menu **Runtime > Change runtime type**: choose the L4.
3. Run the cells from top to bottom. Every line is one iteration; every 4th one has the check against medium (256 games).
4. It runs 100 iterations. My guess is 3 to 3.5 minutes each, so 5 to 6 hours (10 to 12 compute units); the first lines show the real time.
5. After a disconnect run all cells again: it continues where it stopped.
6. When it has ended: **Runtime > Disconnect and delete runtime**.

If the first iteration stops with an error, copy the error text: this design has run on the Mac, but not yet on an Nvidia card.

In [ ]:
# Google Drive: the folder with the finished network
from google.colab import drive
drive.mount('/content/drive')
MODELS = '/content/drive/MyDrive/Super-TikTakToe-models'
import os
os.environ['SUPER_TTT_MODELS'] = MODELS

In [ ]:
# The code from GitHub (the newest version every time)
%cd /content
!test -d Super-TikTakToe || git clone https://github.com/kasparhui/Super-TikTakToe.git
%cd /content/Super-TikTakToe
!git pull

In [ ]:
# Which graphics card did this session get?
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# The training. Stop with the stop button of this cell: everything is saved, and running it again continues.
!python -u src/train.py run large_lines 100

In [ ]:
# The checks of the new network next to large's at the same iterations
import pandas as pd
new = pd.read_csv(f'{MODELS}/alphazero_large_lines_log.csv')
old = pd.read_csv(f'{MODELS}/alphazero_large_log.csv')
new = new[new['vs_medium'].notna()][['iteration', 'vs_medium', 'seconds']].rename(columns={'vs_medium': 'lines', 'seconds': 'seconds_lines'})
old = old[old['vs_medium'].notna()][['iteration', 'vs_medium', 'seconds']].rename(columns={'vs_medium': 'large', 'seconds': 'seconds_large'})
new.merge(old, on='iteration')